In [1]:
from fp2mp_eval.utils import read_json

results = read_json('data/evaluations.json')

In [28]:
from fp2mp_eval.models import Evaluation
from fp2mp_eval import FP2MPEval
import pingouin as pg

for result in results:
    evaluations = [Evaluation(**e) for e in result['evaluations']]
    result['evaluations_df'] = FP2MPEval.evaluations_to_df(evaluations)
    result['evaluations_long_df'] = FP2MPEval.evaluations_to_long_df(evaluations)
    result['icc_df'] = pg.intraclass_corr(result['evaluations_long_df'], targets='indicator', raters='judge', ratings='score')
    result['icc'] = {}
    for _,icc_row in result['icc_df'].iterrows():
        icc_type = icc_row['Type']
        icc_value = icc_row['ICC']
        result['icc'][icc_type] = icc_value
    result['score'] = result['evaluations_df'].mean().mean()

In [18]:
import pandas as pd

problems = {r['problem'] for r in results}
models = {r['model'] for r in results}
icc_types = set(results[0]['icc'].keys())

icc_dfs = {icc_type : pd.DataFrame(0.0, index=list(problems), columns=list(models)) for icc_type in icc_types}
for result in results:
    problem = result['problem']
    model = result['model']
    for icc_type in icc_types:
        icc_dfs[icc_type].loc[problem, model] = result['icc'][icc_type]

# for result in results:

In [24]:
icc_types

{'ICC(1,1)', 'ICC(1,k)', 'ICC(A,1)', 'ICC(A,k)', 'ICC(C,1)', 'ICC(C,k)'}

In [26]:
icc_dfs['ICC(1,1)']

,deepseek/deepseek-v4-pro,deepseek/deepseek-r1
"Город Кемерово. Создай бренд города, основываясь на локальной идентичности.",0.864458,0.895131
"Город Санкт-Петербург. Создай бренд города, основываясь на локальной идентичности.",0.866019,0.718734
Город Санкт-Петербург. Разработай план уплотнения городского центра.,0.681818,0.747375
Город Кемерово. Разработай план уплотнения городского центра.,0.945610,0.642857


In [56]:
data = []

for result in results:
    problem = result['problem']
    model = result['model']
    evaluations_df = result['evaluations_df']
    for judge in evaluations_df.index:
        for indicator in evaluations_df.columns:
            data.append({
                'problem': problem,
                'model': model,
                'judge': judge,
                'indicator': indicator,
                'value': evaluations_df.loc[judge, indicator]
            })

data_df = pd.DataFrame(data)
data_df

,problem,model,judge,indicator,value
0,Город Санкт-Петербург. Разработай план уплотне...,deepseek/deepseek-v4-pro,0,framing,4
1,Город Санкт-Петербург. Разработай план уплотне...,deepseek/deepseek-v4-pro,0,decomposition,4
2,Город Санкт-Петербург. Разработай план уплотне...,deepseek/deepseek-v4-pro,0,diversity,3
3,Город Санкт-Петербург. Разработай план уплотне...,deepseek/deepseek-v4-pro,0,coherence,4
4,Город Санкт-Петербург. Разработай план уплотне...,deepseek/deepseek-v4-pro,0,justification,3
...,...,...,...,...,...
315,"Город Кемерово. Создай бренд города, основывая...",deepseek/deepseek-r1,4,coherence,4
316,"Город Кемерово. Создай бренд города, основывая...",deepseek/deepseek-r1,4,justification,3
317,"Город Кемерово. Создай бренд города, основывая...",deepseek/deepseek-r1,4,uncertainty_handling,2
318,"Город Кемерово. Создай бренд города, основывая...",deepseek/deepseek-r1,4,knowledge_integration,5


In [72]:
data_df.groupby(['model', 'indicator']).agg(
        mean=("value", "mean"),
        std=("value", "std"),
    ).unstack(level=0).reorder_levels([1, 0], axis=1).sort_index(axis=1)

model                 deepseek/deepseek-r1           deepseek/deepseek-v4-pro  \
                                      mean       std                     mean   
indicator                                                                       
coherence                             4.05  0.223607                     4.05   
decomposition                         4.05  0.223607                     4.05   
diversity                             2.75  0.716350                     2.45   
framing                               4.00  0.561951                     3.85   
justification                         3.20  0.615587                     3.15   
knowledge_integration                 4.05  0.686333                     4.20   
metacognition                         2.35  0.587143                     2.30   
uncertainty_handling                  2.35  0.489360                     2.30   

model                            
                            std  
indicator                        
coherence              0.223607  
decomposition          0.394034  
diversity              0.604805  
framing                0.489360  
justification          0.366348  
knowledge_integration  0.410391  
metacognition          0.470162  
uncertainty_handling   0.470162